<a href="https://colab.research.google.com/github/callumhudson0/CIS-3902-AI-ML-Callum-Hudson/blob/main/CH_AI_Project26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bexley Crime Prediction Project

This notebook is a working draft for the project deliverables. It currently focuses on **Deliverable 1: Problem Definition and Data Acquisition**.

## Deliverable 1: Problem Definition and Data Acquisition

### Problem Statement
The aim of this project is to predict the likely **crime type** for Metropolitan Police incidents recorded in the London Borough of **Bexley** using location-based and incident-related variables. The broader motivation is to support crime analysis, identify spatial patterns in offending, and provide a basis for better local decision-making around policing and community safety.

This project is framed as a **supervised classification problem**. The model will use features such as month, approximate location, coordinates, and geographic identifiers to predict the target variable, `Crime type`.

### Target Variable
The target variable is:

- `Crime type`

This variable represents the category of crime recorded by the Met Police, such as anti-social behaviour, vehicle crime, burglary, or violence and sexual offences.

### Use Case / Domain Context
This project sits within the domain of **public safety and crime analytics**. Understanding which types of crime are associated with particular areas can help inform local policing priorities, support community safety planning, as well as improve evidence-based decision-making.

By focusing specifically on Bexley, the project narrows the analysis to a defined borough rather than attempting to model crime patterns across the full Metropolitan Police area.

**NOTE:** The full Met Police csv file was approximately 1.7GB, so running the program would've taken excessive processing power.

### Dataset Selection and Justification
The source data was obtained directly from http://data.police.uk, with crime data spanning from **March 2023 to February 2026**, filtered by the **Metropolitan Police** operating area. The monthly files were provided in two formats:

- `metropolitan-street.csv`
- `metropolitan-outcomes.csv`

All files were initially merged into a single combined dataset so the full structure of the available data could be reviewed. During that review, it became clear that the two source file types used different schemas:

- The **street** files contained variables such as `Crime type`, `Location`, `Longitude`, `Latitude`, and `Last outcome category`.
- The **outcomes** files contained `Outcome type`, but did **not** contain `Crime type`.

Because the objective of this project is to predict `Crime type`, the outcomes-only records were not suitable for the main modeling dataset. For that reason, the full merged data was filtered in two stages:

1. The dataset was filtered to include only records relating to **Bexley**, identified using the `LSOA name` field.
2. A final analysis dataset was created using only the **Bexley street records**, because these contain both the target variable (`Crime type`) and location information.

This final filtered file is the most appropriate dataset for the predictive modeling task.

### Data Acquisition and Preparation Summary
The data acquisition process completed so far is summarised below:

1. Monthly Metropolitan Police CSV files were collected for the period `2023-03` to `2026-02`.
2. The monthly files were merged into a single combined CSV for inspection.
3. The merged file was examined and found to contain two schemas: `street` and `outcomes`.
4. Rows belonging to **Bexley** were extracted into a borough-specific dataset.
5. A second filtered file was created using only **Bexley street records** so that `Crime type` and `Location` were both present.

### Final Dataset for Modeling
The final dataset selected for the project is:

- https://raw.githubusercontent.com/callumhudson0/CIS-3902-AI-ML-Callum-Hudson/main/combined_metropolitan_data_bexley_street_only.csv

This file contains incident records for Bexley only and includes fields suitable for classification modeling, particularly `Crime type` as the response variable and location-related fields as predictors.

### Potential Value of the Model
A predictive model based on this dataset could support decision-making by:

- identifying which crime categories are more likely in certain areas
- highlighting spatial patterns in Bexley crime data
- supporting better allocation of local resources
- providing an evidence base for borough-level safety planning

### Important Data Note
Some rows in the original merged dataset had blank `Crime type` values. This was **not** caused by a merge error. Those rows came from the original `metropolitan-outcomes.csv` files, which do not include a `Crime type` field. This is why the final modeling dataset was restricted to the Bexley **street** records only.

In [8]:
from pathlib import Path

import pandas as pd

data_path = "https://raw.githubusercontent.com/callumhudson0/CIS-3902-AI-ML-Callum-Hudson/main/combined_metropolitan_data_bexley_street_only.csv"
df = pd.read_csv(data_path)
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Outcome type,Context,Source file
0,NaN,2023-03,Metropolitan Police Service,Metropolitan Police Service,0.143210,51.495493,On or near Kencot Way,E01000466,Bexley 001A,Anti-social behaviour,NaN,NaN,NaN,2023-03/2023-03-metropolitan-street.csv
1,NaN,2023-03,Metropolitan Police Service,Metropolitan Police Service,0.125944,51.498757,On or near Hartslock Drive,E01000466,Bexley 001A,Anti-social behaviour,NaN,NaN,NaN,2023-03/2023-03-metropolitan-street.csv
2,NaN,2023-03,Metropolitan Police Service,Metropolitan Police Service,0.125944,51.498757,On or near Hartslock Drive,E01000466,Bexley 001A,Anti-social behaviour,NaN,NaN,NaN,2023-03/2023-03-metropolitan-street.csv
3,NaN,2023-03,Metropolitan Police Service,Metropolitan Police Service,0.131400,51.495556,On or near Kale Road,E01000466,Bexley 001A,Anti-social behaviour,NaN,NaN,NaN,2023-03/2023-03-metropolitan-street.csv
4,NaN,2023-03,Metropolitan Police Service,Metropolitan Police Service,0.131400,51.495556,On or near Kale Road,E01000466,Bexley 001A,Anti-social behaviour,NaN,NaN,NaN,2023-03/2023-03-metropolitan-street.csv


In [9]:
df.shape

(59748, 14)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59748 entries, 0 to 59747
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               48590 non-null  object 
 1   Month                  59748 non-null  object 
 2   Reported by            59748 non-null  object 
 3   Falls within           59748 non-null  object 
 4   Longitude              59748 non-null  float64
 5   Latitude               59748 non-null  float64
 6   Location               59748 non-null  object 
 7   LSOA code              59748 non-null  object 
 8   LSOA name              59748 non-null  object 
 9   Crime type             59748 non-null  object 
 10  Last outcome category  48590 non-null  object 
 11  Outcome type           0 non-null      float64
 12  Context                0 non-null      float64
 13  Source file            59748 non-null  object 
dtypes: float64(4), object(10)
memory usage: 6.4+ MB


In [7]:
df[["Month", "Location", "LSOA code", "LSOA name", "Crime type"]].sample(10, random_state=42)

,Month,Location,LSOA code,LSOA name,Crime type
34047,2024-11,On or near Standard Road,E01000333,Bexley 003A,Anti-social behaviour
16730,2023-12,On or near Edison Road,E01000398,Bexley 029A,Drugs
16382,2023-12,On or near Talehangers Close,E01000389,Bexley 020E,Other theft
31536,2024-09,On or near Braeside Crescent,E01000328,Bexley 017B,Burglary
10476,2023-09,On or near Gayton Road,E01000417,Bexley 006B,Robbery
35697,2024-12,On or near Tavy Bridge,E01000470,Bexley 002F,Criminal damage and arson
5118,2023-06,On or near Station Road North,E01000335,Bexley 003B,Violence and sexual offences
42424,2025-04,On or near Coniston Road,E01000370,Bexley 014D,Anti-social behaviour
52406,2025-10,On or near South Road,E01000431,Bexley 008D,Violence and sexual offences
29149,2024-08,On or near Mildred Road,E01000404,Bexley 004B,Public order
